# Labels Check

Notebook for validating, exploring, and debugging the label pipeline (raw -> LLM -> validated).

In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set up paths
REPO_ROOT = Path("..")
DATA_DIR = REPO_ROOT / "data"
LABELS_DIR = DATA_DIR / "labels"
RAW_DIR = DATA_DIR / "ego4d_data" / "v2" / "annotations"

# Ensure we see all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

print(f"Data dir: {DATA_DIR}")

Data dir: ../data


## Video emulator

In [32]:
# Download video 9b49246f-30e9-476f-ab8c-56a1bcb1d936 locally
!ego4d --output_directory ../data/ego4d_data/videos --datasets full_scale --video_uids 9b49246f-30e9-476f-ab8c-56a1bcb1d936 --yes

Datasets to download: {'full_scale'}
Download Path: ../data/ego4d_data/videos/v2
Ego4D Metadata: ../data/ego4d_data/videos/ego4d.json
Checking requested datasets and versions...
Created download directory for version 'v2_1' of dataset: 'full_scale' at: ../data/ego4d_data/videos/v2/full_scale
Only downloading a subset of the video files because the 'video_uids' flag has been set on the command line or in the config file. A total of 1 video files will be downloaded.

Retrieving object metadata from S3...
100%|███████████████████████████████████████| 1/1 [00:00<00:00, 1410.80object/s]
Checking if latest file versions are already downloaded...
100%|███████████████████████████████████████████| 1/1 [00:00<00:00,  2.71file/s]
No existing videos to filter.
100%|███████████████████████████████████████| 503M/503M [00:17<00:00, 1.86MiB/s]Checking file integrity...
100%|███████████████████████████████████████| 503M/503M [00:17<00:00, 30.3MiB/s]


In [34]:
video_path = DATA_DIR / "ego4d_data" / "videos"/ "v2" / "full_scale" / "9b49246f-30e9-476f-ab8c-56a1bcb1d936.mp4"
if video_path.exists():
    print(f"Video downloaded to: {video_path}")
    display(Video(filename=str(video_path), width=640))
else:
    print("Download failed or path incorrect.")

Video downloaded to: ../data/ego4d_data/videos/v2/full_scale/9b49246f-30e9-476f-ab8c-56a1bcb1d936.mp4


## Data Review


This is a newly created 50 rows _llm file

In [21]:
# Load Full LLM Output (Should have raw columns now)
llm_full_path = LABELS_DIR / "action_labels_llm.csv"
print(f"\nLoading {llm_full_path}...")

if llm_full_path.exists():
    df_llm = pd.read_csv(llm_full_path)
    print(f"Total rows: {len(df_llm)}")
    print("Columns:", df_llm.columns.tolist())
    
    # Display sample
    print("\n--- Action Labels LLM Full (First 5) ---")
    display(df_llm.head(5))
else:
    print("File not found. (Wait for script to finish?)")

# Load Clean LLM Output
llm_clean_path = LABELS_DIR / "action_labels_llm_clean.csv"
print(f"\nLoading {llm_clean_path}...")
if llm_clean_path.exists():
    df_clean = pd.read_csv(llm_clean_path)
    print(f"Total rows: {len(df_clean)}")
    
    print("\n--- Action Labels LLM Clean (First 5) ---")
    display(df_clean.head(5))
else:
    print("File not found.")


Loading ../data/labels/action_labels_llm.csv...
Total rows: 50
Columns: ['video_uid', 'timestamp_sec', 'narration_text', 'scenario', 'action', 'reasoning', 'thinking_process', 'llm_raw_output']

--- Action Labels LLM Full (First 5) ---


,video_uid,timestamp_sec,narration_text,scenario,action,reasoning,thinking_process,llm_raw_output
0,ab9b7b7c-33ad-4358-929f-8b36b719f45f,0.00000,#C C holds a phone.,Walking Outdoors,Stationary,Holding an object with minimal body movement.,NaN,"{\n ""label"": ""Stationary"",\n ""reasoning"": ""Holding an object with minimal body movement.""\n}"
1,ab9b7b7c-33ad-4358-929f-8b36b719f45f,6.93790,#C C walks down a walkway.,Walking Outdoors,Locomotion,"Body moving through space, high body acceleration.",NaN,"{\n ""label"": ""Locomotion"",\n ""reasoning"": ""Body moving through space, high body acceleration.""\n}"
2,ab9b7b7c-33ad-4358-929f-8b36b719f45f,126.33952,#C C climbs a short staircase.,Walking Outdoors,Locomotion,"Body moving through space with climbing action, high body acceleration.",NaN,"{\n ""label"": ""Locomotion"",\n ""reasoning"": ""Body moving through space with climbing action, high body acceleration.""\n}"
3,9b49246f-30e9-476f-ab8c-56a1bcb1d936,5.06864,#C C sings.,Cleaning,Stationary,"Low body/hand motion, singing involves minimal movement.",NaN,"{\n ""label"": ""Stationary"",\n ""reasoning"": ""Low body/hand motion, singing involves minimal movement.""\n}"
4,9b49246f-30e9-476f-ab8c-56a1bcb1d936,6.79379,#C C walks downstairs.,Cleaning,Locomotion,"Body moving through space, indicative of walking.",NaN,"{\n ""label"": ""Locomotion"",\n ""reasoning"": ""Body moving through space, indicative of walking.""\n}"



Loading ../data/labels/action_labels_llm_clean.csv...
Total rows: 50

--- Action Labels LLM Clean (First 5) ---


,video_uid,timestamp_sec,narration_text,scenario,action,reasoning
0,ab9b7b7c-33ad-4358-929f-8b36b719f45f,0.00000,#C C holds a phone.,Walking Outdoors,Stationary,Holding an object with minimal body movement.
1,ab9b7b7c-33ad-4358-929f-8b36b719f45f,6.93790,#C C walks down a walkway.,Walking Outdoors,Locomotion,"Body moving through space, high body acceleration."
2,ab9b7b7c-33ad-4358-929f-8b36b719f45f,126.33952,#C C climbs a short staircase.,Walking Outdoors,Locomotion,"Body moving through space with climbing action, high body acceleration."
3,9b49246f-30e9-476f-ab8c-56a1bcb1d936,5.06864,#C C sings.,Cleaning,Stationary,"Low body/hand motion, singing involves minimal movement."
4,9b49246f-30e9-476f-ab8c-56a1bcb1d936,6.79379,#C C walks downstairs.,Cleaning,Locomotion,"Body moving through space, indicative of walking."


#### Inspect Raw Output

Look at the actual LLM reasoning.

In [22]:
if 'df_llm' in locals() and 'llm_raw_output' in df_llm.columns:
    print("\n--- Sample Raw Output (First Row) ---")
    print(df_llm.iloc[0]['llm_raw_output'])


--- Sample Raw Output (First Row) ---
{
  "label": "Stationary",
  "reasoning": "Holding an object with minimal body movement."
}


Comparison 

In [23]:
# Compare New vs Old Clean
new_clean_path = LABELS_DIR / "action_labels_llm_clean.csv"
old_clean_path = LABELS_DIR / "action_labels_llm_clean.csv.bak_good"

print(f"Loading NEW: {new_clean_path}")
df_new = pd.read_csv(new_clean_path)

print(f"Loading OLD: {old_clean_path}")
# Use nrows to load just the beginning if the file is huge
df_old = pd.read_csv(old_clean_path, nrows=len(df_new)) 

print(f"\nNew Rows: {len(df_new)}")
print(f"Old Rows (loaded): {len(df_old)}")

# Merge on UID + Timestamp to align rows
merged = pd.merge(
    df_new, 
    df_old, 
    on=['video_uid', 'timestamp_sec'], 
    suffixes=('_new', '_old'),
    how='inner'
)

print(f"\nAligned Rows: {len(merged)}")

if not merged.empty:
    # Check for label differences
    diffs = merged[merged['action_new'] != merged['action_old']]
    print(f"\nLabel Mismatches: {len(diffs)} / {len(merged)}")
    
    if not diffs.empty:
        print("\n--- Sample Mismatches ---")
        cols = ['video_uid', 'timestamp_sec', 'narration_text_new', 'action_new', 'action_old']
        display(diffs[cols].head(10))
    else:
        print("\nSUCCESS: All aligned labels match exactly!")
else:
    print("\nWARNING: No matching rows found (UID/Timestamp mismatch). Are the datasets starting at the same point?")

Loading NEW: ../data/labels/action_labels_llm_clean.csv
Loading OLD: ../data/labels/action_labels_llm_clean.csv.bak_good

New Rows: 50
Old Rows (loaded): 50

Aligned Rows: 50

Label Mismatches: 9 / 50

--- Sample Mismatches ---


,video_uid,timestamp_sec,narration_text_new,action_new,action_old
16,9b49246f-30e9-476f-ab8c-56a1bcb1d936,95.742840,#C C closes the dishwasher.,Object Transfer,Essential Operation
18,9b49246f-30e9-476f-ab8c-56a1bcb1d936,104.486220,#C C opens the tap.,Object Transfer,Essential Operation
20,9b49246f-30e9-476f-ab8c-56a1bcb1d936,114.640450,#C C throws the vegetable in the bin.,Locomotion,Object Transfer
21,9b49246f-30e9-476f-ab8c-56a1bcb1d936,118.466170,#C C select the vegetables.,Essential Operation,Object Transfer
27,9b49246f-30e9-476f-ab8c-56a1bcb1d936,212.849340,#C C throws the vegetable in the bin.,Locomotion,Essential Operation
29,9b49246f-30e9-476f-ab8c-56a1bcb1d936,320.206251,#C C throws some of the fruits in the bin.,Object Transfer,Essential Operation
38,9b49246f-30e9-476f-ab8c-56a1bcb1d936,494.295531,#C C adjusts the bowl.,Stationary,Error / Correction
41,9b49246f-30e9-476f-ab8c-56a1bcb1d936,501.134081,#C C opens the fruit.,Object Transfer,Essential Operation
46,9b49246f-30e9-476f-ab8c-56a1bcb1d936,549.608196,#C C opens the tap.,Object Transfer,Essential Operation


#### Verify Source of '#unsure' in scenario labels

Source is indeed from the dataset. Needs to be handled

In [24]:
# Load Raw Narrations Again (specifically looking for the target UID)
target_uid = '9b49246f-30e9-476f-ab8c-56a1bcb1d936'
target_ts = 25.5720096

narration_path = RAW_DIR / "narration.json"
print(f"Checking source data for UID: {target_uid} at {target_ts}s...")

found = False
try:
    with open(narration_path, 'r') as f:
        raw_data = json.load(f)
        
    if target_uid in raw_data:
        # Traverse structure
        vid_data = raw_data[target_uid]
        narrations = []
        if 'narration_pass_1' in vid_data: 
            narrations = vid_data['narration_pass_1']['narrations']
        elif 'narration_pass_2' in vid_data:
            narrations = vid_data['narration_pass_2']['narrations']
            
        # Search for timestamp match (approximate since float)
        for n in narrations:
            # Check if timestamp is close enough (e.g. within 0.001s)
            if abs(n['timestamp_sec'] - target_ts) < 0.001:
                print("\nFOUND MATCH in Source Data:")
                print(json.dumps(n, indent=2))
                found = True
                break
    
    if not found:
        print("Target narration NOT found in source file.")

except Exception as e:
    print(f"Error reading file: {e}")

Checking source data for UID: 9b49246f-30e9-476f-ab8c-56a1bcb1d936 at 25.5720096s...

FOUND MATCH in Source Data:
{
  "timestamp_sec": 25.5720096,
  "timestamp_frame": 767,
  "_unmapped_timestamp_sec": 25.57201,
  "narration_text": "#C C touches camera. #unsure",
  "annotation_uid": "fd1947ce-6647-4a6b-b5c1-a48efb4ba009"
}


#### Print All 50 Rows



In [25]:
# Set display options to show everything
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    if 'df_clean' in locals():
        print("\n--- All 50 Clean Rows ---")
        display(df_clean)
    else:
        print("df_clean not loaded.")


--- All 50 Clean Rows ---


,video_uid,timestamp_sec,narration_text,scenario,action,reasoning
0,ab9b7b7c-33ad-4358-929f-8b36b719f45f,0.000000,#C C holds a phone.,Walking Outdoors,Stationary,Holding an object with minimal body movement.
1,ab9b7b7c-33ad-4358-929f-8b36b719f45f,6.937900,#C C walks down a walkway.,Walking Outdoors,Locomotion,"Body moving through space, high body acceleration."
2,ab9b7b7c-33ad-4358-929f-8b36b719f45f,126.339520,#C C climbs a short staircase.,Walking Outdoors,Locomotion,"Body moving through space with climbing action, high body acceleration."
3,9b49246f-30e9-476f-ab8c-56a1bcb1d936,5.068640,#C C sings.,Cleaning,Stationary,"Low body/hand motion, singing involves minimal movement."
4,9b49246f-30e9-476f-ab8c-56a1bcb1d936,6.793790,#C C walks downstairs.,Cleaning,Locomotion,"Body moving through space, indicative of walking."
5,9b49246f-30e9-476f-ab8c-56a1bcb1d936,25.572010,#C C touches camera. #unsure,Cleaning,Stationary,"Low body/hand motion, likely involves minimal movement."
6,9b49246f-30e9-476f-ab8c-56a1bcb1d936,58.676460,#C C opens a fridge.,Cleaning,Object Transfer,Logistics step involving opening a fridge.
7,9b49246f-30e9-476f-ab8c-56a1bcb1d936,60.201140,#C C takes a pack of vegetables from the fridge. #unsure,Cleaning,Object Transfer,Logistics step involving picking up and moving an object.
8,9b49246f-30e9-476f-ab8c-56a1bcb1d936,69.554770,#C C closes the fridge.,Cleaning,Object Transfer,"Closing an object, a logistics step."
9,9b49246f-30e9-476f-ab8c-56a1bcb1d936,74.981620,#C C moves a bottle on the kitchen counter.,Cleaning,Object Transfer,Logistics step involving moving an object.
